---
title: "Applications of Python for Finance - Week 4 - sample solutions"
author: "Jay & Liam"
date: "2026-09-xx"
format: pdf
---
\newpage
\tableofcontents
\listoffigures
\listoftables
\newpage


# Regression

## Required libraries

Load the required libraries `pandas` , `matplotlib`, and `statsmodels`
in the code cell below.

In [ ]:
#| warning: false

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

## Linear Regression

Linear regression can be used to explain the relationship between values
in the columns of a data set, and whether or not one affects another.

To use the language of linear regression, we can learn whether one
variable (the *response* or *dependent* variable) can be predicted by
some other variable(s) - called *independent* or *explanatory*
variables.

Let's load some data into a *data frame* and see if the values in one
column can predict the values in another column. Load the
`goog_aapl_wk4.csv` file using the code cell below and inspect the
first few rows.

In [ ]:
# load the data and inspect

trading_data = pd.read_csv("https://imperial-business-school.github.io/programming-for-finance/data/goog_aapl_wk4.csv")
print(trading_data.head())

In this example, we'll take an aspect of `Google`'s stock as our
response variable and see if one other explanatory variable in the data
set can predict it. Because we are only looking at one explanatory
variable, this type of regression is known as **simple linear regression**.

### Returns

Rather than looking at the close prices of a stock for analyses that
examine relationships, we'll use returns - the change in value from one
point in time to another point in time.

The formula for daily returns is below. Notice how we can simplify it
further: 

$$
return_{daily} = \frac{(close_{t} - close_{t-1})}{close_{t-1}} 
= \frac {close_{t}} {close_{t-1}}   -1
$$ 

Or if you prefer, the simplified formula in pseudo-code is:

`daily return = (today's close / yesterday's close) - 1`

All of our close prices are in a single column. Given a specific row of
that column, *yesterday's closing price* will always be the row
immediately above it. Recall we can refer to a row above, using `shift()`.


Let's break down the problem, so we have both day and day before in the
same row. This makes it easier to visualise a solution.

In [ ]:
# First lag the close price column into a new column

trading_data["GOOG_Close_Lag"] = trading_data["GOOG.Close"].shift(1)
print(trading_data.head())

# Next apply the daily returns formula
trading_data["GOOG_Returns"] = (trading_data["GOOG.Close"] / trading_data["GOOG_Close_Lag"]) - 1
print(trading_data.head())

##### Exercise 4A: Apple Returns

Calculate daily returns for Apple, into a new column. See if you can do
this in one line of code.

In [ ]:
# Exercise 4A

trading_data["AAPL_Returns"] = (trading_data["AAPL.Close"] / trading_data["AAPL.Close"].shift(1)) - 1

print(trading_data.head())

### Correlation

Before we can move to regression, we should ensure that the response and
explanatory variable are *correlated* - that is, that they share a
linear relationship. If they do not share such a relationship, simple
linear regression may not be the appropriate analysis.

The libraries such as *pandas, matplotlib* can help us determine whether columns in a DataFrame are
*correlated*.

Let's say we thought that *Apple* returns might have some affect on
*Google* returns.

#### Google vs. Apple

In the next code cell, we'll examine the `GOOG_Returns` and `AAPL_Returns` columns of the DataFrame.

In [ ]:
#| warning: false
#| fig-cap: "Correlation: Google vs Apple Returns"

print(trading_data[["GOOG_Returns", "AAPL_Returns"]].corr())

# scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(trading_data["AAPL_Returns"], trading_data["GOOG_Returns"])
plt.xlabel("Apple Returns")
plt.ylabel("Google Returns")
plt.title("Google vs Apple Returns")
plt.show()

The resulting output is a **correlation matrix**.

-   Top-right and Bottom-left are scatter-plots of the two variables, visually
    illustrating their relationship and helping to identify linearity.

-   Top-left and bottom-right we'll see histograms showing the
    distribution of prices for each stock and aim to give a *bigger
    picture* of the data.

**One way** of interpreting coefficients:

-   less than 0.2 means weakly or not correlated
-   between 0.2 and 0.4 means moderately correlated
-   above 0.4 means strongly correlated

What are your thoughts on the Google vs Apple relationship? If there is
a linear correlation, we can move on to regression.

### Fitting a model

To start simple linear regression, we use the `sml.ols().fit()` function
from `statsmodels`. 
There are two important parts:

-   `response variable`.
-   `explanatory variable`, which is used to explain or predict the response.


In [ ]:
# Simple linear regression model

model = smf.ols("GOOG_Returns ~ AAPL_Returns", data=trading_data).fit()
#print(model.summary()) # print all the regression results with model.summary()
print("Coefficients:")
print(round(model.params, 4))

print("\nT-values:")
print(round(model.tvalues, 4))

print("\nP-values:")
print(round(model.pvalues, 4))

print("\nR-squared:")
print(round(model.rsquared, 4))

print("\nAdjusted R-squared:")
print(round(model.rsquared_adj, 4))

print("\nResiduals:")
print(model.resid.head())

A brief explanation of the model summary above:

-   **Residuals** These represent the differences between the observed
    and predicted daily returns of Google.

-   Our **Coefficients**:

    -   **Intercept** is the predicted daily return for Google when
        Apple's return is 0%.
    -   **AAPL.Returns** indicates that For every 1 percentage point
        increase in Apple's daily return, Google's daily return is
        predicted to increase by approximately 0.7493 percentage points.

-   The **t-value** & **p-value** validate the significance of the
    coefficient. A high t-value / low p-value mean that the coefficient
    is likely significant.

-   **R-squared** tells us that 52.63% of the variability in Google's
    daily returns can be explained by Apple's daily returns. The
    Adjusted R-square is *adjusted* for the number of predictors, which
    is useful for models with multiple predictors.

The model can be interpreted in the format of $y = a + bx$

$$ GOOG.Return = -0.001096 + 0.7493 * AAPL.Return $$

If we are pleased with our model, we can draw a regression line on a
plot of our data.

##### Exercise 4B: Plotting

Plot Google daily returns vs. Apple daily returns, with the response
variable on the y-axis. If needed, filter out NA values - here
we'll use `dropna()`.

We can obtain the fitted values from the regression model with `model.fittedvalues`, then save 
them into the DataFrame, sort the data by Apple returns for a cleaner line,
and then use the `plt.plot()` function
to add our regression line to our plot.


In [ ]:
#| fig-cap: "Regression: Apple VS Google"

# Add the fitted values to the original data frame for plotting
trading_data["fitted"] = model.fittedvalues
# Sort the data by Apple Returns for a cleaner plot
plot_data = trading_data.sort_values("AAPL_Returns")

plt.figure(figsize=(8, 5))

plt.scatter(plot_data["AAPL_Returns"], plot_data["GOOG_Returns"], label="Data")
plt.plot(plot_data["AAPL_Returns"], plot_data["fitted"], color="red", label="Regression Line")
plt.xlabel("Apple Returns")
plt.ylabel("Google Returns")
plt.title("Google vs Apple Returns")
plt.legend()
plt.show()

## Multiple Linear Regression

As the name implies, multiple linear regression determines if *more than
one* explanatory variable is involved with predicting a dependent
variable.

Today we'll work with an established model in portfolio management
called the *Fama French 3-Factor* model, after its two creators (Fama &
French) and three accepted market risk factors. We've provided Fama
French 3-factor (FF3) data for *ff3_wk4.csv* file. Let's load that data
now and have a look at it.

In [ ]:
# load data with FF3

ff3_data = pd.read_csv("https://imperial-business-school.github.io/programming-for-finance/data/ff3_wk4.csv")
ff3_data = ff3_data.rename(columns={"Mkt.RF": "Mkt_RF"})

**IMPORTANT NOTE** FF3 factors are in percentages, but we'll need them
in their decimal for our further analyses. We can convert multiple
columns like so:

In [ ]:
# This piece of code takes all the columns except the first (Date)
# and divides them by 100

ff3_data[["Mkt_RF", "SMB", "HML", "RF"]] = ff3_data[["Mkt_RF", "SMB", "HML", "RF"]] / 100

### Merging Dataframes

For a basic merge, we provide the names of two data frames as arguments
to `pd.merge()`. We also provide the `on` argument, to indicate *the column
we want to merge on*.

The *column to merge on* must have the same column name, type and format
in both dataframes. Fortunately for us, this is already the case.

In [ ]:
# Merging dataframes

merged_data = pd.merge(trading_data, ff3_data, on="Date")


### Excess Returns

Let's continue using Google's data for practice. Multiple regression
with FF3 requires **excess returns** as the response variable. To get
excess returns, we first calculate simple daily returns. From the daily
returns, we subtract the current risk free rate - an approximate
indicator of an investment free of risk, often a short-term government
bond - to get what we call excess returns.

Using the code cell below, create a new column in the data frame called
`GOOG_ExcessReturns`, correctly populating it with excess returns.

In [ ]:
# Excess returns

merged_data["GOOG_ExcessReturns"] = merged_data["GOOG_Returns"] - merged_data["RF"]

### Fitting our model

The pseudo-code formula for fitting our FF3 model is: 
$$
y = Mkt.RF + SMB + HML
$$

In [ ]:
# Multiple regression model

ff3_model = smf.ols("GOOG_ExcessReturns ~ Mkt_RF + SMB + HML", data=merged_data).fit()
#print(ff3_model.summary())
print("Coefficients:")
print(round(ff3_model.params, 4))

print("\nT-values:")
print(round(ff3_model.tvalues, 4))

print("\nP-values:")
print(round(ff3_model.pvalues, 4))

print("\nR-squared:")
print(round(ff3_model.rsquared, 4))

print("\nAdjusted R-squared:")
print(round(ff3_model.rsquared_adj, 4))

print("\nResiduals:")
print(model.resid.head())

Let's look at the three variables we're using to explain Google's
performance:

-   **Mkt.RF (Market Risk Premium)** If this coefficient is positive, it
    suggests that the stock tends to move in the same direction as the
    wider market (and vice versa). A positive coefficient greater than 1
    indicates higher volatility than the market, while a positive
    coefficient less than 1 indicates lower volatility than the market.

-   **SMB (Small Minus Big)** A positive, significant coefficient here
    suggests that the stock's excess returns behave more like those of
    small-cap stocks, (even if Google itself is a large-cap company.)

-   **HML (High Minus Low)** A positive coefficient suggests the stock
    behaves more like a "value" stock (think stability - Coca-Cola,
    Berkshire Hathaway). A negative coefficient implies it behaves more
    like a "growth" stock (think growth - Amazon, Tesla)

The model can be interpreted in the format of
$y = a + b_1x_1 + b_2x_2+b_3x_3$ 

$$
GOOG.ExcessReturns = -0.0001211 + 1.1045703 * Mkt.RF - 0.6693132 * SMB - 0.3870928 * HML
$$

## Programming Structure

A frequently used structure is a combination of loop and condition.
First, let's try a `for` loop, which **iterates** over any vector type
data.

Let's loop over some vectors

In [ ]:
# Loop over a sequence
for i in range(1, 7):
    print(i)

# Introduce the next loop
print("next loop...")

# Mini-Exercise: Using a loop, print 1, 10, 100, 1000
for i in [1, 10, 100, 1000]:
    print(i)

Loop over a dataframe using its index

In [ ]:
# Index loops over the first 10 rows of the Google closing price
for i in range(10): 
  print(trading_data["GOOG.Close"][i])

#### Exercise 4C: Loop and Conditions

Loop over a sequence from 0 to 20 and print "WOW" for each odd number

In [ ]:
# Exercise 4C: loop and conditionals
for i in range(0, 21):
    if i % 2:
        print("WOW")

# Introduce the next example

for i in range(21):
    if i % 2:
        print(i, "WOW")

Looping over *iterables* (i.e. objects that can be looped!)


In [ ]:
# Our object Google close price is iterable so we can loop over those too!
for price in trading_data["GOOG.Close"]: 
  print(price)
  # That's a lot of rows... This is a good place for a conditional (and a break)!
  if price > 145:
    break
  

## Render to PDF

Now, *Render to PDF* again and check the output file.

## The End

That's all for this week.

**But please read the supplement below!**

# Supplement: Data cleaning

In reality, the data will be **messy**. Some examples of **messy**
are: - Additional unrelated data in our CSV files - Non-matching column
names or types - Differences in data formats

Data frames will rarely match up as nicely as you have seen above, so
the following steps show how we achieve the above during the data
cleaning stage.

### Loading Messy Data

Current FF3 data can be found at [the following
link](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

For this exercise, we'll use another trading data and FF3 data: -
"another_trading_wk4.csv" - "F-F_Research_Data_Factors_daily.CSV"

Loading the trading data is straightforward:


In [ ]:
# Load trading data

trading = pd.read_csv("https://imperial-business-school.github.io/programming-for-finance/data/another_trading_wk4.csv")

In [ ]:
trading.head()

In [ ]:
# drop the column with the index in it
trading = trading.drop(columns=["Unnamed: 0"])
trading.head()

Running the code below should load the CSV file. What happens instead?

In [ ]:
# Loading FF3

ff3 = pd.read_csv("https://imperial-business-school.github.io/programming-for-finance/data/F-F_Research_Data_Factors_daily.CSV")

Open the FF3 `.CSV` file in VS Code and see if you can find the issue.

The `pd.read_csv()` function accepts an additional argument called `skiprows`.
**Add it to the code in the cell above to skip the first four rows.**

Then use the code cell below to inspect the last rows of your FF3 data
frame. What do you notice?

Looks like some copyright information made its way into the table. Here
are a few approaches to fix it - pick one and try it below to get some
cleaner data.

-   the `pd.read_csv()` function has a numeric argument called `skiprows` you
    can use
-   the `dropna()` function returns a copy of your data frame with all
    NA removed


In [ ]:
# Load FF3 data again

ff3 = pd.read_csv("https://imperial-business-school.github.io/programming-for-finance/data/F-F_Research_Data_Factors_daily.CSV", skiprows = 4)


ff3 = ff3.dropna()


In [ ]:
ff3.head()

#### Don't Forget! Converting FF3 percentages to decimal

Remember that our FF3 factor data comes in percentages, that we'll need
in the decimal for our multiple regression.

In [ ]:
# Converting to decimal
# previously we used
# ff3_data[["Mkt_RF", "SMB", "HML", "RF"]] = ff3_data[["Mkt_RF", "SMB", "HML", "RF"]] / 100

# another method is
ff3.iloc[:, 1:] = ff3.iloc[:, 1:] / 100
ff3.head()

### Cleaning Column Data

As we saw, to merge different dataframes, we need column names, types
and date formats to all match up.

#### Renaming columns

To rename columns, use `rename()`, vector indexing and variable
assignment. Let's rename the *ff3 data* date column to match the
*trading data* date column

In [ ]:
# Rename the first column in the FF3 data to Date

ff3 = ff3.rename(columns={ff3.columns[0]: "Date"})
ff3.head()

#### Formatting date columns

The importance of converting dates to the Date type is of the utmost
here. Remember back to our work with the `pd.to_datetime()` function and
convert date columns in both the FF3 and trading data to the Date type.
This ensures the types and formats will match up for a merge.

In [ ]:
trading.head()

In [ ]:
trading.info()

In [ ]:
# Convert trading data
trading["Date"] = pd.to_datetime(trading["Date"], format="%Y-%m-%d")
trading.head()
# Convert ff3 data
ff3["Date"] = pd.to_datetime(ff3["Date"], format="%Y%m%d")
ff3.head()

#### Subsetting our data

To prepare for our FF3 multiple regression, we'll want to just select
the Google subset of our trading data. To do so, we can use a condition
in our indexing.

In [ ]:
# subsetting rows

goog = trading[trading["Ticker"] == "GOOG"]
goog.head()

Now that we've just got Google data, we don't need the Ticker column.

In [ ]:
# subsetting columns

goog = goog.iloc[:, [0, 2]]
goog.head()

#### Merging dataframes

Finally, we are ready to merge the trading data with the FF3 data.

In [ ]:
# Merging clean dataframes

merged = pd.merge(goog, ff3, on="Date")
merged.head()

Then we would normally proceed with calculating *daily returns* and
*excess returns*, and then fitting our model!